# OpenCV in Practice, M1: fix a badly exposed photo

Every code block in the posts, in order, so you can run them on your own machine. On the
site they run in the browser; here they run on the OpenCV you install below.

Frames from Mark Fairchild's HDR Photographic Survey, "Luxo Double Checker" (research and non-commercial use), developed by this repository's `practice-m1`.

In [ ]:
# Pinned so a Colab base-image change cannot silently move the numbers.
%pip install -q 'opencv-python>=5.0.0,<6.0.0' 'numpy>=2' 'matplotlib>=3.8'

In [ ]:
import os
import urllib.request

import cv2
import matplotlib.pyplot as plt

print("OpenCV", cv2.__version__)

# The module's images, served by condados.ai. Cloudflare refuses Python's
# default user agent, so the request names itself.
BASE = os.environ.get("PRACTICE_BASE", "https://condados.ai")
UA = {"User-Agent": "fundamentals-lab/0.1 (+https://github.com/CondadosAI/fundamentals-lab)"}
for path in ['/practice/m1/dark.webp', '/practice/m1/reference.webp', '/practice/m1/bright.webp', '/practice/m1/darker.webp']:
    name = path.rsplit("/", 1)[-1]
    if not os.path.exists(name):
        req = urllib.request.Request(BASE + path, headers=UA)
        open(name, "wb").write(urllib.request.urlopen(req).read())
print("images:", 'dark.webp', 'reference.webp', 'bright.webp', 'darker.webp')

In [ ]:
# cv2.imshow opens a window in a script and does nothing useful in a notebook.
# Draw with matplotlib instead (OpenCV is BGR, matplotlib expects RGB).
def _imshow(title, img):
    plt.figure(figsize=(10, 6))
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")
    plt.show()

cv2.imshow = _imshow
cv2.waitKey = lambda delay=0: -1

## Open an Image and Read Its Pixels in OpenCV

https://condados.ai/blog/opencv-read-image-pixels

In [ ]:
import cv2

img = cv2.imread("dark.webp")
print("shape:", img.shape)
print("dtype:", img.dtype)
cv2.imshow("dark.webp", img)

In [ ]:
# One pixel: row 320, column 480. OpenCV stores B, G, R.
print("pixel at row 320, column 480:", img[320, 480])

# The whole image in one number: its mean brightness
grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("mean brightness:", round(grey.mean(), 1), "of 255")

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# Pretend another library read the array as R, G, B
swapped = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.imshow("blue and red swapped", swapped)

**Your turn.** Convert to grey with cv2.cvtColor, then ask the array for its largest value.

In [ ]:
brightest = 0  # replace with the brightest grey value in img
print(brightest)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(brightest == int(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).max()))
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Image Histograms in OpenCV

https://condados.ai/blog/opencv-image-histogram

In [ ]:
import cv2
import numpy as np

img = cv2.imread("dark.webp")
grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hist = cv2.calcHist([grey], [0], None, [256], [0, 256])

low = 100 * hist[:64].sum() / grey.size
high = 100 * hist[250:].sum() / grey.size
print("darkest quarter (0-63):", round(low, 1), "%")
print("250 or above:", round(high, 1), "%")

In [ ]:
def draw_histogram(grey, height=200):
    hist = cv2.calcHist([grey], [0], None, [256], [0, 256]).ravel()
    canvas = np.full((height, 256, 3), 255, np.uint8)
    for value, count in enumerate(hist / hist.max()):
        top = int(height * (1 - count))
        cv2.line(canvas, (value, height), (value, top), (80, 80, 80))
    return canvas

ref = cv2.cvtColor(cv2.imread("reference.webp"), cv2.COLOR_BGR2GRAY)
cv2.imshow("dark (left) and reference (right)",
           np.hstack([draw_histogram(grey), draw_histogram(ref)]))

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# The range excludes its upper end: [0, 255] drops 255
wrong = cv2.calcHist([grey], [0], None, [256], [0, 255]).ravel()
right = cv2.calcHist([grey], [0], None, [256], [0, 256]).ravel()
print("pixels at 255, range [0, 255]:", int(wrong[255]))
print("pixels at 255, range [0, 256]:", int(right[255]))

**Your turn.** Read bright.webp, convert it to grey, and take the share of pixels that are 250 or more, as a percentage.

In [ ]:
# percent of grey pixels at 250 or above in bright.webp
clipped = 0.0
print(clipped)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(abs(clipped - 100 * (cv2.cvtColor(cv2.imread('bright.webp'), cv2.COLOR_BGR2GRAY) >= 250).mean()) < 0.1)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Brightness and Contrast in OpenCV

https://condados.ai/blog/opencv-brightness-contrast

In [ ]:
import cv2
import numpy as np

img = cv2.imread("dark.webp")
ref = cv2.imread("reference.webp")

def distance(a, b):
    # mean gap per pixel and channel, on the 0-255 scale
    return np.abs(a.astype(int) - b.astype(int)).mean()

added = cv2.add(img, np.full_like(img, 50))
multiplied = cv2.convertScaleAbs(img, alpha=2.4)
print("as it is:  ", round(distance(img, ref), 1))
print("add 50:    ", round(distance(added, ref), 1))
print("times 2.4: ", round(distance(multiplied, ref), 1))

In [ ]:
def half(x):
    return cv2.resize(x, None, fx=0.5, fy=0.5)

small = [half(x) for x in (added, multiplied, ref)]
cv2.imshow("add 50 | times 2.4 | reference", np.hstack(small))

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# NumPy + on uint8 wraps around instead of stopping at 255
wrapped = img + np.full_like(img, 50)
y, x = np.unravel_index(img.max(axis=2).argmax(), img.shape[:2])
print("brightest pixel:", img[y, x])
print("numpy:", wrapped[y, x], " cv2.add:", added[y, x])
cv2.imshow("img + 50 with NumPy", wrapped)

**Your turn.** The bright photo needs a multiplier below 1. Try values around 0.5.

In [ ]:
bright = cv2.imread("bright.webp")
fixed = cv2.convertScaleAbs(bright, alpha=1.0)  # change alpha
print(round(distance(fixed, ref), 1))

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(distance(fixed, ref) < 30)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Gamma Correction in OpenCV with a Lookup Table

https://condados.ai/blog/opencv-gamma-correction

In [ ]:
import cv2
import numpy as np

ref = cv2.imread("reference.webp")

def distance(a, b):
    return np.abs(a.astype(int) - b.astype(int)).mean()

def gamma(img, g):
    table = (np.arange(256) / 255) ** g * 255
    table = table.round().astype(np.uint8)
    return cv2.LUT(img, table)

for name, g, a in [("dark.webp", 0.55, 2.4),
                   ("darker.webp", 0.40, 4.2)]:
    img = cv2.imread(name)
    by_gamma = distance(gamma(img, g), ref)
    by_gain = distance(cv2.convertScaleAbs(img, alpha=a), ref)
    print(name, "gamma", g, round(by_gamma, 1),
          "| times", a, round(by_gain, 1))

In [ ]:
darker = cv2.imread("darker.webp")
times = cv2.convertScaleAbs(darker, alpha=4.2)
for name, x in [("as shot", darker), ("times 4.2", times),
                ("gamma 0.40", gamma(darker, 0.40))]:
    share = (x.max(axis=2) == 255).mean()
    print(name, "- at 255:", round(100 * share, 1), "%")
def half(x):
    return cv2.resize(x, None, fx=0.5, fy=0.5)

small = [half(x) for x in (times, gamma(darker, 0.40), ref)]
cv2.imshow("times 4.2 | gamma 0.40 | reference", np.hstack(small))

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# Raising 0-255 values to a power without scaling to 0-1 first
wrong = darker.astype(float) ** 0.40
wrong = np.clip(wrong, 0, 255).astype(np.uint8)
print("largest value after the wrong gamma:", wrong.max())
cv2.imshow("gamma on 0-255 instead of 0-1", wrong)

**Your turn.** A gamma above 1 darkens. Try values between 2 and 3.

In [ ]:
# choose a gamma that darkens bright.webp toward the target
g = 1.0
print(round(distance(gamma(cv2.imread("bright.webp"), g), ref), 1))

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(distance(gamma(cv2.imread('bright.webp'), g), ref) < 36)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Histogram Equalization and CLAHE in OpenCV

https://condados.ai/blog/opencv-histogram-equalization

In [ ]:
import cv2
import numpy as np

img = cv2.imread("dark.webp")
ref = cv2.imread("reference.webp")

def distance(a, b):
    return np.abs(a.astype(int) - b.astype(int)).mean()

def on_brightness(img, op):
    # equalize brightness only: Y of YCrCb, colour untouched
    ycc = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    ycc[..., 0] = op(ycc[..., 0])
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)

equalized = on_brightness(img, cv2.equalizeHist)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
local = on_brightness(img, clahe.apply)
print("equalizeHist:", round(distance(equalized, ref), 1))
print("CLAHE:       ", round(distance(local, ref), 1))

In [ ]:
def half(x):
    return cv2.resize(x, None, fx=0.5, fy=0.5)

small = [half(x) for x in (equalized, local, ref)]
cv2.imshow("equalizeHist | CLAHE | reference", np.hstack(small))

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# equalizeHist takes one 8-bit channel, not a colour image
cv2.equalizeHist(img)

**Your turn.** Convert img to grey first, then call apply on the CLAHE object.

In [ ]:
out = None  # CLAHE on the grey version of img
print(out)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(out.ndim == 2 and out.dtype == np.uint8 and out.shape == img.shape[:2])
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Crop, Resize and Save an Image in OpenCV

https://condados.ai/blog/opencv-crop-resize-save

In [ ]:
import os
import cv2

fixed = cv2.convertScaleAbs(cv2.imread("dark.webp"), alpha=2.4)
chart = fixed[150:450, 0:330]  # rows 150-449, columns 0-329
thumb = cv2.resize(fixed, (480, 320), interpolation=cv2.INTER_AREA)

cv2.imwrite("fixed.jpg", fixed, [cv2.IMWRITE_JPEG_QUALITY, 90])
cv2.imwrite("fixed.png", fixed)
for name in ("fixed.jpg", "fixed.png"):
    print(name, os.path.getsize(name) // 1024, "KB")
cv2.imshow("the crop", chart)
cv2.imshow("the thumbnail", thumb)

**Break it.** The common mistake, run on purpose. Some raise an error.

In [ ]:
# cv2.resize takes (width, height); img.shape is (height, width)
h, w = fixed.shape[:2]
squashed = cv2.resize(fixed, (h // 2, w // 2))
print("asked for", (h // 2, w // 2), "got shape", squashed.shape)
cv2.imshow("width and height swapped", squashed)

**Your turn.** Work out the height from the width: fixed.shape[0] * 320 / fixed.shape[1], then round it.

In [ ]:
small = fixed  # resize to 320 pixels wide, keeping the proportions
print(small.shape)

In [ ]:
# Checks your answer; prints 'not yet' until the cell above is done.
try:
    ok = bool(small.shape[1] == 320 and abs(small.shape[0] - round(fixed.shape[0] * 320 / fixed.shape[1])) <= 1)
except Exception:
    ok = False
print("correct" if ok else "not yet")

## Fix a Badly Exposed Photo with OpenCV

https://condados.ai/blog/fix-badly-exposed-photo-opencv

In [ ]:
import cv2
import numpy as np

img = cv2.imread("dark.webp")
ref = cv2.imread("reference.webp")

fixed = cv2.convertScaleAbs(img, alpha=2.4)
gap = np.abs(fixed.astype(int) - ref.astype(int)).mean()
print("distance to the reference:", round(gap, 1))

cv2.imwrite("fixed.jpg", fixed, [cv2.IMWRITE_JPEG_QUALITY, 90])
side = [cv2.resize(x, None, fx=0.5, fy=0.5) for x in (img, fixed, ref)]
cv2.imshow("dark | fixed | reference", np.hstack(side))